<a href="https://colab.research.google.com/github/qk8015-lgtm/114-2-ProgramingLanguage/blob/main/HW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil

In [72]:
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

In [73]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [74]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
# 注意：執行時請點擊儲存格下方的「Allow (允許)」按鈕以授權存取金鑰
try:
    api_key = userdata.get('gemini')
    # 使用獲取的金鑰配置 genai
    genai.configure(api_key=api_key)
    # 使用更穩定的 1.5 系列模型
    model = genai.GenerativeModel('gemini-1.5-flash')
    print("✅ API 金鑰讀取成功！")
except Exception as e:
    print(f"❌ 讀取金鑰失敗：{str(e)}")
    print("💡 請確保左側 Secrets 面板中的 'gemini' 已開啟 'Notebook access' 並在執行時點擊『允許』。")

❌ 讀取金鑰失敗：Requesting secret gemini timed out. Secrets can only be fetched when running from the Colab UI.
💡 請確保左側 Secrets 面板中的 'gemini' 已開啟 'Notebook access' 並在執行時點擊『允許』。


In [75]:
import google.generativeai as genai
from google.colab import userdata

try:
    # 嘗試重新獲取金鑰並配置
    api_key = userdata.get('gemini')
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-1.5-flash')

    print("🔍 正在測試 API 連線...")
    test_response = model.generate_content('你好，請回覆「連線成功」')
    print(f"✅ API 連線測試成功！ AI 回覆：{test_response.text}")
except Exception as e:
    print(f"❌ 診斷失敗：{str(e)}")
    print("💡 請確保：1. Secrets 名稱是 gemini  2. 已勾選 Notebook access 3. 執行時點擊了『允許』權限請求")

❌ 診斷失敗：Requesting secret gemini timed out. Secrets can only be fetched when running from the Colab UI.
💡 請確保：1. Secrets 名稱是 gemini  2. 已勾選 Notebook access 3. 執行時點擊了『允許』權限請求


In [76]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sa2n02HTV0YGSzoM4XJyjpj4pm5tklrC76p6okxCveY/edit?usp=sharing"
WORKSHEET_NAME = "工作表3"
TIMEZONE = "Asia/Taipei"

In [77]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url(SHEET_URL)


# 從 gsheets 的 All-whiteboard-device 載入 sheets
sh = gsheets.worksheet(WORKSHEET_NAME).get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sh[1:], columns=sh[0])
# 取得最前面的5筆資料
df.head()


""


In [78]:

sh = gc.open_by_url(SHEET_URL)


In [110]:
import time
import uuid
import re
import json
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import google.generativeai as genai
from google.colab import userdata

# 基礎配置
sh = gc.open_by_url(SHEET_URL)

def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    return ws

TASKS_HEADER = ["id","task","status","priority","est_min","start_time","end_time","actual_min","pomodoros","due_date","labels","notes","created_at","updated_at","completed_at","planned_for"]
LOGS_HEADER = ["log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"]

ws_tasks = ensure_worksheet(sh, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(sh, "pomodoro_logs", LOGS_HEADER)

# 初始化 AI 模型
try:
    current_key = userdata.get('gemini')
    genai.configure(api_key=current_key)
    ai_model = genai.GenerativeModel('gemini-2.0-flash-exp')
except:
    ai_model = None

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    data = ws.get_all_values()
    if not data or len(data) <= 1: return pd.DataFrame(columns=header)
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

def write_df(ws, df, header):
    ws.clear()
    ws.update([header] + df.astype(str).values.tolist() if not df.empty else [header])

def add_task(name, pri):
    if not name: return
    new_row = [str(uuid.uuid4())[:8], name, "todo", pri, 25, "", "", 0, 0, "", "", "", tznow().isoformat(), tznow().isoformat(), "", ""]
    ws_tasks.append_row(new_row)

def log_pomodoro(task_id, phase, minutes):
    _now = tznow()
    _start = (_now - timedelta(minutes=minutes)).isoformat()
    new_log = {
        "log_id": str(uuid.uuid4())[:8], "task_id": task_id, "phase": phase,
        "start_ts": _start, "end_ts": _now.isoformat(), "minutes": round(minutes, 2),
        "cycles": 1, "note": ""
    }
    df_logs = read_df(ws_logs, LOGS_HEADER)
    write_df(ws_logs, pd.concat([df_logs, pd.DataFrame([new_log])], ignore_index=True), LOGS_HEADER)
    df_t = read_df(ws_tasks, TASKS_HEADER)
    idx = df_t.index[df_t['id'] == task_id]
    if len(idx) > 0:
        df_t.at[idx[0], 'actual_min'] = int((pd.to_numeric(df_t.at[idx[0], 'actual_min'], errors='coerce') or 0) + minutes)
        if phase == 'work':
            df_t.at[idx[0], 'pomodoros'] = int((pd.to_numeric(df_t.at[idx[0], 'pomodoros'], errors='coerce') or 0) + 1)
        write_df(ws_tasks, df_t, TASKS_HEADER)

def ai_plan_tasks(user_prompt):
    if not ai_model: return "(AI 目前無法連線)"
    df = read_df(ws_tasks, TASKS_HEADER)
    if df.empty: return "(目前無任務資料)"
    tasks_info = df[['id', 'task', 'priority', 'status']].to_string()
    prompt = f"指令: {user_prompt}\n\n目前任務清單:\n{tasks_info}\n\n請以專業助手的角度提供計畫。請用繁體中文。"
    try:
        return ai_model.generate_content(prompt).text
    except:
        return "(Gemini API 額度已達上限，暫時無法產生建議)"

def scrape_and_parse(url):
    if not url: return "⚠️ 請輸入網址"
    try:
        res = requests.get(url, timeout=10)
        res.encoding = res.apparent_encoding
        soup = BeautifulSoup(res.text, 'html.parser')
        title = soup.title.string if soup.title else "無標題"
        text = soup.get_text(separator=' ', strip=True)[:1500]
        if not ai_model: return "(AI 模型未啟動)"
        prompt = f"請從內容擷取待辦事項: 網址 {url}, 標題 {title}, 內容 {text}。輸出 JSON: {{'has_task': bool, 'task_name': str, 'due_date': str, 'priority': 'H'|'M'|'L', 'notes': str}}"
        response = ai_model.generate_content(prompt)
        match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if not match: return "(AI 分析失敗)"
        data = json.loads(match.group())
        if data.get('has_task'):
            ws_tasks.append_row([str(uuid.uuid4())[:8], data['task_name'], "todo", data['priority'], 25, "", "", 0, 0, data['due_date'], "課程爬蟲", f"來源: {url}\n{data['notes']}", tznow().isoformat(), tznow().isoformat(), "", ""])
            return f"✅ 已新增任務: {data['task_name']}"
        return "ℹ️ 未偵測到待辦事項。"
    except:
        return "(抓取或分析過程發生錯誤)"

def _refresh():
    t = read_df(ws_tasks, TASKS_HEADER)
    l = read_df(ws_logs, LOGS_HEADER)
    choices = [(f"[{r['status']}] {r['task']}", r['id']) for _, r in t.iterrows()] if not t.empty else []
    return t, l, gr.update(choices=choices), gr.update(choices=choices)

def start_timer(task_id, duration):
    if not task_id: yield "⚠️ 請先選擇任務", gr.update(interactive=True); return
    rem = int(duration * 60)
    while rem > 0:
        mins, secs = divmod(rem, 60)
        yield f"⏳ {mins:02d}:{secs:02d}", gr.update(interactive=False)
        time.sleep(1); rem -= 1
    log_pomodoro(task_id, "work", duration)
    yield "🔔 專注結束!", gr.update(interactive=True)

with gr.Blocks() as demo:
    gr.Markdown("# 🍅 任務管理與 AI 課程公告爬蟲")
    with gr.Tab("自動擷取 (Web Import)"):
        url_in = gr.Textbox(label="課程公告網址"); btn_scrape = gr.Button("🔍 抓取並自動產生任務", variant="primary"); scrape_msg = gr.Markdown()
    with gr.Tab("Tasks"):
        with gr.Row():
            t_in = gr.Textbox(label="任務名稱"); p_in = gr.Dropdown(["H","M","L"], value="M", label="優先級"); btn_add = gr.Button("➕ 新增")
        grid_t = gr.Dataframe(label="任務清單")
    with gr.Tab("Pomodoro"):
        sel_t = gr.Dropdown(label="🎯 執行任務"); work_len = gr.Slider(1, 60, value=25, step=1, label="分鐘"); timer_out = gr.Label("Ready"); btn_go = gr.Button("🚀 開始專注", variant="primary")
    with gr.Tab("Logs"): grid_l = gr.Dataframe(label="歷史紀錄")
    with gr.Tab("API & AI"):
        ai_in = gr.Textbox(label="AI 指令"); ai_btn = gr.Button("✨ 產生建議"); ai_out = gr.Markdown()

    btn_scrape.click(lambda u: [scrape_and_parse(u), *_refresh()], url_in, [scrape_msg, grid_t, grid_l, sel_t, sel_t])
    btn_add.click(lambda n, p: [add_task(n,p), *_refresh()][1:], [t_in, p_in], [grid_t, grid_l, sel_t, sel_t])
    btn_go.click(start_timer, [sel_t, work_len], [timer_out, btn_go])
    ai_btn.click(ai_plan_tasks, ai_in, ai_out)
    demo.load(_refresh, None, [grid_t, grid_l, sel_t, sel_t])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7834a2ef0d74594347.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import google.generativeai as genai
from google.colab import userdata
import gspread
from google.auth import default

print("--- 系統最終確認 ---")

# 1. 驗證 Gemini API 狀態
try:
    key = userdata.get('gemini')
    genai.configure(api_key=key)
    model = genai.GenerativeModel('gemini-1.5-flash')
    res = model.generate_content('Hi', generation_config={'max_output_tokens': 5})
    print(f"✅ Gemini API: 運作中 (回應: {res.text.strip()})")
except Exception as e:
    print(f"❌ Gemini API 存取失敗: {e}")

# 2. 驗證試算表與必要分頁
try:
    creds, _ = default()
    gc = gspread.authorize(creds)
    target_sh = gc.open_by_url(SHEET_URL)
    titles = [ws.title for ws in target_sh.worksheets()]
    print(f"✅ 試算表連線成功！")
    print(f"📊 目前分頁清單: {titles}")

    required = ["tasks", "pomodoro_logs"]
    for r in required:
        if r in titles:
            print(f"  - 分頁 '{r}': 已就緒")
        else:
            print(f"  - 分頁 '{r}': 尚未建立 (系統執行時會自動初始化)")
except Exception as e:
    print(f"❌ 試算表存取失敗: {e}")